In [1]:
import duckdb
import pandas as pd

In [2]:
feat_con = duckdb.connect(database=":memory:")

In [4]:
feat_con.query(""" CREATE TABLE dpbl_duck AS SELECT * FROM read_csv_auto('data/duckdb_cleaned_dblp.csv')""")
feat_con.query(""" CREATE TABLE train_duck AS SELECT * FROM read_csv_auto('data/train.csv')""")

In [5]:
feat_con.query("""SELECT * FROM dpbl_duck""").show()

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┬─────────┬────────────────────────────────┬──────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────┬─────────────────────────────────────────────────────────────────┬──────────────────────┬───────────────┐
│                                                        pauthor                                                        │                                                                  ptitle                                                                   │ pyear │   pid   │              pkey              │                 journal                  │                                  journalfull                          

In [6]:
feat_con.query("""SELECT * FROM train_duck""").show()

┌─────────┬──────────────────────────────┬───────────────────────────────────┬─────────┬───────────┐
│ column0 │             key1             │               key2                │  label  │ partition │
│  int64  │           varchar            │              varchar              │ boolean │   int64   │
├─────────┼──────────────────────────────┼───────────────────────────────────┼─────────┼───────────┤
│       0 │ conf/prib/AhmedF07           │ journals/jcc/PatraS09             │ false   │         7 │
│       1 │ conf/vlsid/ChenCC95          │ journals/tcad/LuoCWCCW08          │ true    │         4 │
│       2 │ conf/prozess/Sun88           │ conf/isnn/SunZLCS07               │ true    │         8 │
│       3 │ conf/pricai/BeaumontTSM04    │ conf/icip/SattarAS08              │ false   │         5 │
│       4 │ conf/dft/SemiaoRVSTT07       │ conf/iolts/Rodriguez-IragoAVSTT05 │ true    │         7 │
│       6 │ conf/lmo/DemphlousL96        │ conf/reflection/DemphlousL99      │ true    │   

## JOIN

In [8]:
feat_con.query("""CREATE OR REPLACE TABLE train_pairs AS
                SELECT t.*,
                d1.pauthor AS pauthor_1,
                d1.ptitle AS ptitle_1,
                d1.pyear AS pyear_1,
                d1.ptype AS ptype_1,
                d1.journal AS journal_1,
                d1.journalfull AS journalfull_1,
                d1.booktitle AS booktitle_1,
                d1.booktitlefull AS booktitlefull_1,
               
                d2.pauthor AS pauthor_2,
                d2.ptitle AS ptitle_2,
                d2.pyear AS pyear_2,
                d2.ptype AS ptype_2,
                d2.journal AS journal_2,
                d2.journalfull AS journalfull_2,
                d2.booktitle AS booktitle_2,
                d2.booktitlefull AS booktitlefull_2
                FROM train_duck t
               
               LEFT JOIN dpbl_duck d1 ON t.key1 = d1.pkey
               LEFT JOIN dpbl_duck d2 ON t.key2 = d2.pkey""")

In [9]:
feat_con.query("""SELECT * FROM train_pairs""").show()

┌─────────┬─────────────────────────────────────┬────────────────────────────────────────┬─────────┬───────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬───────────────┬─────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────┬────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬───────────────┬───────────────────────────────────────────────────────┬────────────────────────────────────────────────────────

In [ ]:
pandas_df = feat_con.query("""SELECT * FROM train_pairs""").to_df()

### TODO: check if baseline works similarly in terms of performance

## Title Similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")
tfidf = vectorizer.fit_transform(df["ptitle"].fillna(""))
key_to_index = {k: i for i, k in enumerate(df["pkey"])}

def title_similarity(row):
    i = key_to_index.get(row["key1"])
    j = key_to_index.get(row["key2"])
    
    if i is None or j is None:
        return 0
    
    return cosine_similarity(tfidf[i], tfidf[j])[0][0]

train_pairs["title_sim"] = train_pairs.apply(title_similarity, axis=1)

## Author Similarity

## Year Difference

## Book Title

## Journal Title

## P_Type